In [1]:
import glob
import os
import stim
import numpy as np
import pickle
from tqdm import tqdm
from css import compute_css_logical_operators
from css_simulator import construct_css_resource_state
import time
# # ── Error rate / shot configuration ──────────────────────────────────────────
# error_rates      = np.linspace(1.0, 0.0, 20, endpoint=False)
# shot_counts      = [3000 * (2 ** i) for i in range(20)]
# error_shots_dict = dict(zip(error_rates, shot_counts))

# # ── Collect all HGP code files ────────────────────────────────────────────────
# code_dir   = "/Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/ECC_gen/code_lib/hgp_code_lib/"
# code_files = sorted(glob.glob(code_dir + "hgp_*.pkl"))

# tqdm.write(f"{'='*60}")
# tqdm.write(f"Found {len(code_files)} code files to process.")
# tqdm.write(f"Error rates : {len(error_shots_dict)} levels from {max(error_rates):.3f} → {min(error_rates):.3f}")
# tqdm.write(f"Shot counts : {min(shot_counts):,} → {max(shot_counts):,}")
# tqdm.write(f"{'='*60}\n")

# # ── Outer loop: codes ─────────────────────────────────────────────────────────
# for code_idx, code_path in enumerate(tqdm(code_files, desc="Codes", position=0, leave=True), start=1):

#     code_start_time        = time.time()
#     base                   = os.path.basename(code_path)
#     _, n_str, k_str, d_str = base.replace(".pkl", "").split("_")
#     n_code, k_code, d_code = int(n_str), int(k_str), int(d_str)
#     tag                    = f"{n_code}_{k_code}_{d_code}"

#     tqdm.write(f"\n[Code {code_idx}/{len(code_files)}] {base}")
#     tqdm.write(f"  Parameters : [[n={n_code}, k={k_code}, d={d_code}]]")
#     tqdm.write(f"  Rate       : {k_code/n_code:.4f}")

#     # ── Load matrices ─────────────────────────────────────────────────────────
#     tqdm.write(f"  Loading matrices ...")
#     with open(code_path, "rb") as f:
#         code = pickle.load(f)

#     hgp_x = code["hgp_x"].astype(np.uint8, copy=False)
#     hgp_z = code["hgp_z"].astype(np.uint8, copy=False)
#     tqdm.write(f"  Hx shape   : {hgp_x.shape}  |  Hz shape: {hgp_z.shape}")

#     # ── Compute logical operators ─────────────────────────────────────────────
#     tqdm.write(f"  Computing logical operators ...")
#     logical_x_matrix, logical_z_matrix = compute_css_logical_operators(hgp_x, hgp_z)
#     num_logical_qubits, num_physical_qubits = logical_x_matrix.shape
#     num_total_qubits = num_physical_qubits + num_logical_qubits
#     tqdm.write(f"  Physical qubits : {num_physical_qubits}")
#     tqdm.write(f"  Logical  qubits : {num_logical_qubits}")
#     tqdm.write(f"  Total    qubits : {num_total_qubits}")

#     # ── Save logical operators ────────────────────────────────────────────────
#     operators_path = f"operators_{tag}.pkl"
#     tqdm.write(f"  Saving logical operators to {operators_path} ...")
#     with open(operators_path, "wb") as f:
#         pickle.dump({
#             "hgp_x":           hgp_x,
#             "hgp_z":           hgp_z,
#             "logical_x":       logical_x_matrix,
#             "logical_z":       logical_z_matrix,
#             "n":               n_code,
#             "k":               k_code,
#             "d_est":           d_code,
#         }, f)
#     tqdm.write(f"  Saved: {operators_path}")

#     # ── Build base circuit ────────────────────────────────────────────────────
#     tqdm.write(f"  Building base circuit ...")
#     circuit_start = time.time()
#     tableau       = construct_css_resource_state(
#         hgp_x.toarray(),
#         hgp_z.toarray(),
#         logical_x_matrix.toarray(),
#         logical_z_matrix.toarray()
#     )
#     circuit_init      = (tableau + tableau).to_circuit()
#     circuit_build_time = time.time() - circuit_start
#     tqdm.write(f"  Circuit built in {circuit_build_time:.2f}s : {len(circuit_init)} instructions")

#     # ── Save base circuit ─────────────────────────────────────────────────────
#     circuit_path = f"circuit_{tag}.pkl"
#     tqdm.write(f"  Saving base circuit to {circuit_path} ...")
#     with open(circuit_path, "wb") as f:
#         pickle.dump({
#             "circuit_init":        circuit_init,
#             "num_physical_qubits": num_physical_qubits,
#             "num_logical_qubits":  num_logical_qubits,
#             "num_total_qubits":    num_total_qubits,
#             "circuit_build_time":  circuit_build_time,
#             "n":                   n_code,
#             "k":                   k_code,
#             "d_est":               d_code,
#         }, f)
#     tqdm.write(f"  Saved: {circuit_path}")

#     # ── Inner loop: error rates ───────────────────────────────────────────────
#     tqdm.write(f"  Starting sampling over {len(error_shots_dict)} error rates ...")
#     results_dict       = {}
#     total_shots_so_far = 0

#     for depol_error_rate, num_shots in tqdm(
#         error_shots_dict.items(),
#         desc=f"  [[{n_code},{k_code},{d_code}]] error rates",
#         position=1,
#         leave=False
#     ):
#         shot_start = time.time()
#         circuit    = circuit_init.copy()
#         circuit.append("DEPOLARIZE1", list(range(num_physical_qubits)), depol_error_rate)

#         for qubit_index in range(num_total_qubits):
#             target_qubit = qubit_index + num_total_qubits
#             circuit.append("MXX", [qubit_index, target_qubit])
#             circuit.append("MZZ", [qubit_index, target_qubit])

#         tqdm.write(f"    Sampling : depol={depol_error_rate:.4f}  shots={num_shots:>12,} ...")
#         sampler       = circuit.compile_sampler()
#         measurements  = np.array(sampler.sample(shots=num_shots), dtype=np.uint8)
#         shot_duration = time.time() - shot_start
#         total_shots_so_far += num_shots

#         results_dict[depol_error_rate] = {
#             "num_shots":         num_shots,
#             "measurements":      measurements,
#             "sampling_time_sec": shot_duration,
#         }
#         tqdm.write(f"    Done     : depol={depol_error_rate:.4f}  shots={num_shots:>12,}  "
#                    f"shape={measurements.shape}  "
#                    f"time={shot_duration:.2f}s  "
#                    f"cumulative={total_shots_so_far:,}")

#     # ── Save samples ──────────────────────────────────────────────────────────
#     sample_path = f"sample_{tag}.pkl"
#     tqdm.write(f"  Saving samples to {sample_path} ...")
#     with open(sample_path, "wb") as f:
#         pickle.dump(results_dict, f)
#     tqdm.write(f"  Saved: {sample_path}")

#     # ── Save metadata ─────────────────────────────────────────────────────────
#     meta_path      = f"meta_{tag}.pkl"
#     code_duration  = time.time() - code_start_time
#     tqdm.write(f"  Saving metadata to {meta_path} ...")
#     with open(meta_path, "wb") as f:
#         pickle.dump({
#             "n":                    n_code,
#             "k":                    k_code,
#             "d_est":                d_code,
#             "rate":                 k_code / n_code,
#             "hgp_x_shape":          hgp_x.shape,
#             "hgp_z_shape":          hgp_z.shape,
#             "num_physical_qubits":  num_physical_qubits,
#             "num_logical_qubits":   num_logical_qubits,
#             "num_total_qubits":     num_total_qubits,
#             "error_rates":          list(error_shots_dict.keys()),
#             "shot_counts":          list(error_shots_dict.values()),
#             "total_shots":          total_shots_so_far,
#             "circuit_build_time":   circuit_build_time,
#             "total_time_sec":       code_duration,
#             "source_file":          code_path,
#             "timestamp":            time.strftime("%Y-%m-%d %H:%M:%S"),
#         }, f)
#     tqdm.write(f"  Saved    : {meta_path}")
#     tqdm.write(f"  Total time for this code : {code_duration:.2f}s")
#     tqdm.write(f"[Code {code_idx}/{len(code_files)}] COMPLETE ✓")

# tqdm.write(f"\n{'='*60}")
# tqdm.write(f"All {len(code_files)} codes processed successfully.")
# tqdm.write(f"{'='*60}")

In [5]:
# ── Error rate / shot configuration ──────────────────────────────────────────
error_rates      = np.linspace(1.0, 0.0, 20, endpoint=False)
shot_counts      = [3000 * (2 ** i) for i in range(20)]
error_shots_dict = dict(zip(error_rates, shot_counts))

# ── Collect all HGP code files ────────────────────────────────────────────────
code_dir   = "/Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/ECC_gen/code_lib/hgp_code_lib2/"
code_files = sorted(glob.glob(code_dir + "hgp_*.pkl"))

tqdm.write(f"{'='*60}")
tqdm.write(f"Found {len(code_files)} code files.")
tqdm.write(f"{'='*60}\n")

Found 11 code files.



In [6]:
tqdm.write("PHASE 1: Computing logical operators\n")

for code_idx, code_path in enumerate(tqdm(code_files, desc="Logical Operators", position=0, leave=True), start=1):

    base                   = os.path.basename(code_path)
    _, n_str, k_str, d_str = base.replace(".pkl", "").split("_")
    n_code, k_code, d_code = int(n_str), int(k_str), int(d_str)
    tag                    = f"{n_code}_{k_code}_{d_code}"

    tqdm.write(f"\n[{code_idx}/{len(code_files)}] {base}")

    # Load matrices
    with open(code_path, "rb") as f:
        code = pickle.load(f)

    hgp_x = code["hgp_x"].astype(np.uint8, copy=False)
    hgp_z = code["hgp_z"].astype(np.uint8, copy=False)
    tqdm.write(f"  Hx shape : {hgp_x.shape}  |  Hz shape : {hgp_z.shape}")

    # Compute logical operators
    tqdm.write(f"  Computing logical operators ...")
    start            = time.time()
    logical_x_matrix, logical_z_matrix = compute_css_logical_operators(hgp_x, hgp_z)
    num_logical_qubits, num_physical_qubits = logical_x_matrix.shape
    num_total_qubits = num_physical_qubits + num_logical_qubits
    tqdm.write(f"  Done in {time.time() - start:.2f}s")
    tqdm.write(f"  Physical : {num_physical_qubits}  |  Logical : {num_logical_qubits}  |  Total : {num_total_qubits}")

    # Save
    operators_path = f"operators_{tag}.pkl"
    with open(operators_path, "wb") as f:
        pickle.dump({
            "hgp_x":                hgp_x,
            "hgp_z":                hgp_z,
            "logical_x":            logical_x_matrix,
            "logical_z":            logical_z_matrix,
            "num_physical_qubits":  num_physical_qubits,
            "num_logical_qubits":   num_logical_qubits,
            "num_total_qubits":     num_total_qubits,
            "n":                    n_code,
            "k":                    k_code,
            "d_est":                d_code,
        }, f)
    tqdm.write(f"  Saved : {operators_path}")

tqdm.write(f"\nPHASE 1 COMPLETE ✓ — {len(code_files)} operator files saved.\n")

PHASE 1: Computing logical operators



Logical Operators:   0%|          | 0/11 [00:00<?, ?it/s]


[1/11] hgp_10000_400_16.pkl
  Hx shape : (4800, 10000)  |  Hz shape : (4800, 10000)
  Computing logical operators ...


Logical Operators:   9%|▉         | 1/11 [04:20<43:26, 260.66s/it]

  Done in 260.63s
  Physical : 10000  |  Logical : 400  |  Total : 10400
  Saved : operators_10000_400_16.pkl

[2/11] hgp_2500_100_12.pkl
  Hx shape : (1200, 2500)  |  Hz shape : (1200, 2500)
  Computing logical operators ...


...

  Done in 113.64s
  Physical : 7225  |  Logical : 289  |  Total : 7514
  Saved : operators_7225_289_16.pkl

[10/11] hgp_8100_324_16.pkl
  Hx shape : (3888, 8100)  |  Hz shape : (3888, 8100)
  Computing logical operators ...


Logical Operators:  91%|█████████ | 10/11 [12:47<01:36, 96.29s/it]

  Done in 143.26s
  Physical : 8100  |  Logical : 324  |  Total : 8424
  Saved : operators_8100_324_16.pkl

[11/11] hgp_9025_361_16.pkl
  Hx shape : (4332, 9025)  |  Hz shape : (4332, 9025)
  Computing logical operators ...


Logical Operators: 100%|██████████| 11/11 [16:21<00:00, 89.21s/it] 

  Done in 213.57s
  Physical : 9025  |  Logical : 361  |  Total : 9386
  Saved : operators_9025_361_16.pkl

PHASE 1 COMPLETE ✓ — 11 operator files saved.



In [ ]:
tqdm.write("PHASE 2: Building base circuits\n")

operator_files = sorted(glob.glob("operators_*.pkl"))
tqdm.write(f"Found {len(operator_files)} operator files.\n")

for code_idx, op_path in enumerate(tqdm(operator_files, desc="Base Circuits", position=0, leave=True), start=1):

    base                   = os.path.basename(op_path)
    _, n_str, k_str, d_str = base.replace(".pkl", "").split("_")
    n_code, k_code, d_code = int(n_str), int(k_str), int(d_str)
    tag                    = f"{n_code}_{k_code}_{d_code}"

    tqdm.write(f"\n[{code_idx}/{len(operator_files)}] {base}")

    # Load operators
    with open(op_path, "rb") as f:
        ops = pickle.load(f)

    hgp_x              = ops["hgp_x"]
    hgp_z              = ops["hgp_z"]
    logical_x_matrix   = ops["logical_x"]
    logical_z_matrix   = ops["logical_z"]
    num_physical_qubits = ops["num_physical_qubits"]
    num_logical_qubits  = ops["num_logical_qubits"]
    num_total_qubits    = ops["num_total_qubits"]
    tqdm.write(f"  Loaded operators for [[n={n_code}, k={k_code}, d={d_code}]]")

    # Build circuit
    tqdm.write(f"  Building base circuit ...")
    start   = time.time()
    tableau = construct_css_resource_state(
        hgp_x.toarray(),
        hgp_z.toarray(),
        logical_x_matrix.toarray(),
        logical_z_matrix.toarray()
    )
    circuit_init       = (tableau + tableau).to_circuit()
    circuit_build_time = time.time() - start
    tqdm.write(f"  Built in {circuit_build_time:.2f}s  |  Instructions : {len(circuit_init)}")

    # Save
    circuit_path = f"circuit_{tag}.pkl"
    with open(circuit_path, "wb") as f:
        pickle.dump({
            "circuit_init":         circuit_init,
            "num_physical_qubits":  num_physical_qubits,
            "num_logical_qubits":   num_logical_qubits,
            "num_total_qubits":     num_total_qubits,
            "circuit_build_time":   circuit_build_time,
            "n":                    n_code,
            "k":                    k_code,
            "d_est":                d_code,
        }, f)
    tqdm.write(f"  Saved : {circuit_path}")

tqdm.write(f"\nPHASE 2 COMPLETE ✓ — {len(operator_files)} circuit files saved.\n")

PHASE 2: Building base circuits

Found 16 operator files.



Base Circuits:   0%|          | 0/16 [00:00<?, ?it/s]